This notebook demonstrates how to use the BiomechZoo library to analyze movement data collected from IMU sensors.
The analysis includes data preprocessing, event detection, gait cycle partitioning, normalization, and visualization of the results.

Data source: The data used in this notebook is from the "Losing" dataset, which contains IMU data collected from participants performing
various walking tasks. The data were modified for the purposes of this tutorial.

The full dataset is available at: https://doi.org/10.6084/m9.figshare.c.5758997

The dataset used here is avaialbe in https://github.com/mcgillmotionlab/biomechzoo/tree/main/data/losing/raw

The dataset is described in the following paper: Losing, V., Hasenjäger, M. A Multi-Modal Gait Database of Natural Everyday-Walk in an Urban Environment. Sci Data 9, 473 (2022). https://doi.org/10.1038/s41597-022-01580-3

In [20]:
# import relevant biomech modules
%load_ext autoreload
%autoreload 2
import os
from biomechzoo.biomechzoo import BiomechZoo
from biomechzoo.utils.find_repo_root import find_repo_root

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
# Step 0: get raw data folder and initialize biomechzoo object
project_root = find_repo_root(os.getcwd())
fld_raw_data = os.path.join(project_root, 'data', 'losing', 'raw')
inplace = False  # tells biomechzoo to create new folders instead of writing to the original folder
verbose = 2 # minimal output to the user
name_excludes = ['section_1', 'section_3'] # these trials won't be processed
bmech = BiomechZoo(fld_raw_data, verbose=verbose, inplace=inplace)

BiomechZoo initialized
verbosity set to: 2
root processing folder set to: /Users/phild/Code/biomechzoo/data/losing/raw
Processing mode: backup (inplace=False)(each step will be applied to a new folder)


In [22]:
# step 1: convert to zoo
# frequency is 60 Hz as per the database documentation
bmech.table2zoo(extension='csv', freq=60, out_folder='1-csv2zoo')

converting .csv to zoo for /Users/phild/Code/biomechzoo/data/losing/raw/trial_001_p1_taskA_sec01.csv
table2zoo process complete for 1 file(s) in 0.02 secs
all files saved to: /Users/phild/Code/biomechzoo/data/losing/1-csv2zoo


In [23]:
# step 2 clean
ch_remove = ['angularVelocity_Pelvis_x', 'angularVelocity_Pelvis_y','angularVelocity_Pelvis_z', 'walk_mode', 'section',
             'time', 'participant_id']
ch_old = ['jointAngle_jRightKnee_x', 'jointAngle_jRightKnee_y', 'jointAngle_jRightKnee_z']
ch_new = ['RightKneeAngle_x', 'RightKneeAngle_y', 'RightKneeAngle_z']
bmech.removechannel(ch=ch_remove, out_folder='2-clean')
bmech.renamechannel(ch_current=ch_old, ch_new=ch_new, out_folder='2-clean', inplace=True)

removing channels for /Users/phild/Code/biomechzoo/data/losing/1-csv2zoo/honda_walk_subset_p1_section_1.zoo
Removing channels: ['time', 'participant_id', 'angularVelocity_Pelvis_x', 'angularVelocity_Pelvis_y', 'angularVelocity_Pelvis_z', 'walk_mode', 'section']
removing channels for /Users/phild/Code/biomechzoo/data/losing/1-csv2zoo/honda_walk_subset_p1_section_11.zoo
Removing channels: ['time', 'participant_id', 'angularVelocity_Pelvis_x', 'angularVelocity_Pelvis_y', 'angularVelocity_Pelvis_z', 'walk_mode', 'section']
removing channels for /Users/phild/Code/biomechzoo/data/losing/1-csv2zoo/honda_walk_subset_p1_section_13.zoo
Removing channels: ['time', 'participant_id', 'angularVelocity_Pelvis_x', 'angularVelocity_Pelvis_y', 'angularVelocity_Pelvis_z', 'walk_mode', 'section']
removing channels for /Users/phild/Code/biomechzoo/data/losing/1-csv2zoo/honda_walk_subset_p1_section_15.zoo
Removing channels: ['time', 'participant_id', 'angularVelocity_Pelvis_x', 'angularVelocity_Pelvis_y', '

In [24]:
# inspect the file
bmech.file_info()

File: /Users/phild/Code/biomechzoo/data/losing/2-clean/honda_walk_subset_p1_section_17.zoo
Size: 619840 bytes
Video frequency: 60 Hz
Analog frequency: {} Hz
Channels (13): task, acceleration_Pelvis_x, acceleration_Pelvis_y, acceleration_Pelvis_z, acceleration_RightLowerLeg_x, acceleration_RightLowerLeg_y, acceleration_RightLowerLeg_z, angularVelocity_RightLowerLeg_x, angularVelocity_RightLowerLeg_y, angularVelocity_RightLowerLeg_z, RightKneeAngle_x, RightKneeAngle_y, RightKneeAngle_z
Samples: 4036
First 5 sample(s):
sample	task	acceleration_Pelvis_x	acceleration_Pelvis_y	acceleration_Pelvis_z	acceleration_RightLowerLeg_x	acceleration_RightLowerLeg_y	acceleration_RightLowerLeg_z	angularVelocity_RightLowerLeg_x	angularVelocity_RightLowerLeg_y	angularVelocity_RightLowerLeg_z	RightKneeAngle_x	RightKneeAngle_y	RightKneeAngle_z
0	A	-2.14144	0.659704	4.236152	-5.667929	8.425031	2.733259	1.517975	1.949884	1.309121	-1.574289	9.408164	8.335534
1	A	-1.004549	2.207016	4.719607	-8.316984	9.917188	-

In [26]:
# step 3: get foot strikes from Kielmat algorithm
bmech.compute_magnitude(ch_1='acceleration_Pelvis_x', ch_2='acceleration_Pelvis_y', ch_3='acceleration_Pelvis_z', ch_new='acceleration_Pelvis_magnitude', out_folder='3-processing', inplace=True)
bmech.addevent(event_type='kielmat_fs', event_name='foot_strike', ch='acceleration_Pelvis_magnitude', out_folder='3-processing', inplace=True)

compute magnitude from channels acceleration_Pelvis_x, acceleration_Pelvis_y, acceleration_Pelvis_z for /Users/phild/Code/biomechzoo/data/losing/2-clean/honda_walk_subset_p1_section_1.zoo
compute magnitude from channels acceleration_Pelvis_x, acceleration_Pelvis_y, acceleration_Pelvis_z for /Users/phild/Code/biomechzoo/data/losing/2-clean/honda_walk_subset_p1_section_11.zoo
compute magnitude from channels acceleration_Pelvis_x, acceleration_Pelvis_y, acceleration_Pelvis_z for /Users/phild/Code/biomechzoo/data/losing/2-clean/honda_walk_subset_p1_section_13.zoo
compute magnitude from channels acceleration_Pelvis_x, acceleration_Pelvis_y, acceleration_Pelvis_z for /Users/phild/Code/biomechzoo/data/losing/2-clean/honda_walk_subset_p1_section_15.zoo
compute magnitude from channels acceleration_Pelvis_x, acceleration_Pelvis_y, acceleration_Pelvis_z for /Users/phild/Code/biomechzoo/data/losing/2-clean/honda_walk_subset_p1_section_17.zoo
compute magnitude from channels acceleration_Pelvis_x, a

ValueError: `initial` must be `None` or `0`.

In [ ]:
# step 4: partition by gait cycle and normalize
bmech.split_trial_by_gait_cycle(first_event_name='foot_strike_1', out_folder='4-split', inplace=False)
bmech.normalize()

In [ ]:
# inspect one file
bmech.plot(ch='RightKneeAngle_z', file_path='honda_walk_subset_p1_6.zoo')


In [ ]:
# step 7 plot all files
import os
# import statements
from biomechzoo.ensembler.ensembler import Ensembler
from biomechzoo.ensembler.plot_spec import PlotSpec
from biomechzoo.ensembler.renderers import IndividualLinesRenderer, MeanSDRenderer, CompositeRenderer
from biomechzoo.ensembler.helpers import ConditionSpec, ConditionSource

spec = ConditionSpec(source = ConditionSource.BETWEEN,
                     conditions = ['running'])
channels = ['acc_mag']
rows = 1
cols = 2

lines_and_events = CompositeRenderer(IndividualLinesRenderer())     # within stuff

fig = (
    Ensembler(in_folder=bmech.in_folder,  channels=channels, n_rows=rows,  n_cols=cols, condition_spec=spec)
    .add_subplot(PlotSpec(channel='acc_mag', condition = 'running', companions = [], row=1, col=1, renderer=lines_and_events))
    .add_subplot(PlotSpec(channel='acc_mag', condition = 'running', companions = [], row=1, col=2, renderer=MeanSDRenderer()))
    .build(title=' running')
)
fig.show()


